# Cleaning downloaded Data from GISAID

Author: Alexander Maksiaev

Purpose: Download and clean GISAID data, after de-duplicating from Andersen/NCBI Virus data

Notes: 
* The "downloads" folder MUST be your computer's downloads folder, or wherever your browser automatically downloads files. This folder must also be cleaned in between each run of this code.
* The returned files from this code will be stored in a separate folder after running -- no other action is needed, aside from cleaning the original downloads folder after this code runs.  
* This file MUST be in the same folder as "utils.py"

## Housekeeping ##

In [ ]:
import os
import shutil
import pandas as pd
import numpy as np
import dateutil
import openpyxl
from itertools import islice
import importlib
import utils  
importlib.reload(utils)
from utils import * 

pd.options.mode.chained_assignment = None # suppress warnings when using slices to make new columns

In [2]:
# Paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# downloads = "C:/Users/maksi/Downloads/"
# references = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/references"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references"
downloads = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder. Clean out downloads folder after each use. 
andersen_ncbi_virus_gisaid = home + "Combinations/NCBI_Virus_Andersen_GISAID/" 

# Collect user input

# locations = input("Locations (separate with commas and no spaces): ")
# start_date = input("Start date (format: YYYY-MM-DD): ")
# end_date = input("End date (format: YYYY-MM-DD): ")

locations = "Antarctica,North America,South America"
start_date = "2021-11-01"
end_date = "2026-01-16"
date_range = dateutil.parser.parse(start_date).strftime("%m-%d-%Y") + "--" + dateutil.parser.parse(end_date).strftime("%m-%d-%Y")

os.chdir(downloads)

# Create directories if needed
downloads_saved = home + "GISAID/downloads/" + start_date + "--" + end_date + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
andersen_ncbi_virus = home + "Combinations/NCBI_Virus_Andersen/" + dateutil.parser.parse(start_date).strftime("%m-%d-%Y") + "--" + dateutil.parser.parse(end_date).strftime("%m-%d-%Y") + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
gisaid_files = home + "GISAID/complete/" + start_date + "--" + end_date + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
complete_files = andersen_ncbi_virus_gisaid + dateutil.parser.parse(start_date).strftime("%m-%d-%Y") + "--" + dateutil.parser.parse(end_date).strftime("%m-%d-%Y") + "_" + locations.replace(",", "_").replace(" ", "_") + "/"

if not os.path.exists(gisaid_files): # checking if the directory exists or not
    os.makedirs(gisaid_files) # if the directory is not present then create it

if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it

# Get list of genotypes and states

# os.chdir("C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/references/")
os.chdir(references)

states = pd.read_csv("states_ref.csv")

# genotypes_df = pd.read_excel("genotype_key.xlsx")

# genotypes = list(genotypes_df["Genotype"])
# genotypes.append("Unassigned")

# print(genotypes)

genotypes = ["B3.13", "D1.1", "D1.3"] 

## Download all files, convert fasta files to dataframes

In [5]:
all_metadata_files = []
all_fasta_files = []

if not os.path.exists(downloads_saved): # checking if the directory exists or not
    os.makedirs(downloads_saved) # if the directory is not present then create it

# Move downloaded files to saved downloads
for dirpath, dirs, files in os.walk(downloads):
    if len(files) > 0: # If we have any files that need to be moved
        for file in files:
            file_name = os.path.join(dirpath, file)
            destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
            try:
                shutil.move(file_name, destination_path)
            except:
                print("Error moving file", file_name)
                continue 
    elif len(files) == 0 and len(os.listdir(downloads_saved)) == 0: # If we don't have any downloaded files
        # Have user type in username and password
        username = input("Username: ")
        password = input("Password: ")
        browser = input("Browser: ")
        sleep_time = input("Seconds to sleep in between clicks (recommended 5): ")

        open_gisaid(username, password, browser, sleep_time, locations, start_date, end_date) # Download files -- NOT WORKING RIGHT NOW

        # Re-try 
        for dirpath, dirs, files in os.walk(downloads):
            if len(files) > 0: # If we have any files that need to be moved
                for file in files:
                    file_name = os.path.join(dirpath, file)
                    destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
                    try:
                        shutil.move(file_name, destination_path)
                    except:
                        print("Error moving file", file_name)
                        continue 
            break 
    else: # If we have downloaded files saved already
        continue
    break 

# If we have multiple files, name them nicely
for dirpath, dirs, files in os.walk(downloads_saved):
    for file in files:
        file_name = os.path.join(dirpath, file)
        # file_name = "_".join(file_name.split(" "))
        
        
        os.rename(file_name, "_".join(file_name.split(" ")).replace("(", "").replace(")", ""))
        file_name = "_".join(file_name.split(" ")).replace("(", "").replace(")", "")

        print(file_name)

        # Now go through files and get contents
        if ".xls" in file_name:
            metadata = pd.read_excel(file_name)
            all_metadata_files.append(metadata)
        if ".fasta" in file_name:
            fasta_file = fasta_df(file_name, states) # Convert fasta file to dataframe
            all_fasta_files.append(fasta_file)
    break 

C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/GISAID/downloads/2021-11-01--2026-01-16_Antarctica_North_America_South_America/gisaid_epiflu_isolates.xls
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/GISAID/downloads/2021-11-01--2026-01-16_Antarctica_North_America_South_America/gisaid_epiflu_isolates_1.xls
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/GISAID/downloads/2021-11-01--2026-01-16_Antarctica_North_America_South_America/gisaid_epiflu_sequence_1.fasta
C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/GISAID/downloads/2021-11-01--2026-01-16_Antarctica_North_America_South_America/gisaid_epiflu_sequence.fasta


In [39]:
# Concatenate metadata

metadata_concat = pd.DataFrame()
for metadata_file in all_metadata_files:
    metadata_concat = pd.concat([metadata_concat, metadata_file])

In [42]:
# Separate fastas by segment
segment_fastas = []
unique_animals_all = []
for i, fasta in enumerate(all_fasta_files):

    metadata = metadata_concat

    unique_animals = sort_animals(fasta) # Find unique animals
    # print("Animals: ", unique_animals)
    unique_animals_all.append(unique_animals)

    os.chdir(references)
    animals_ref = pd.read_csv("animals_ref.csv")

    fastas, unique_segments = separate_fasta_by_segs(metadata, fasta, animals_ref, genotypes) # Separate the fasta dataframes into 8 different files based on segment

    for fasta in fastas:
        segment_fastas.append(fasta)

print(segment_fastas) # [0][0][segment_fastas[0][0]["Genotype"] == "D1.1"])

B3.13
0        EPI_ISL_20310330
1        EPI_ISL_20310330
2        EPI_ISL_20310330
3        EPI_ISL_20310330
4        EPI_ISL_20310330
               ...       
12019    EPI_ISL_20307658
12020    EPI_ISL_20307658
12021    EPI_ISL_20307658
12022    EPI_ISL_20307658
12023    EPI_ISL_20307658
Name: Identifier, Length: 12024, dtype: object
                                                  Header  Isolate_Id  \
2912   EPI_ISL_20307361|A/dairy_cow/California/02121-...   02121-017   
2913   EPI_ISL_20307361|A/dairy_cow/California/02121-...   02121-017   
2914   EPI_ISL_20307361|A/dairy_cow/California/02121-...   02121-017   
2915   EPI_ISL_20307361|A/dairy_cow/California/02121-...   02121-017   
2916   EPI_ISL_20307361|A/dairy_cow/California/02121-...   02121-017   
...                                                  ...         ...   
10315  EPI_ISL_20307632|A/dairy_cow/Nebraska/025430-0...  025430-001   
10316  EPI_ISL_20307632|A/dairy_cow/Nebraska/025430-0...  025430-001   
10317  EPI_IS

## De-Duplication

In [7]:
# Get files from Andersen and NCBI Virus

# Grab files
andersen_ncbi = {}
for dirpath, dirs, files in os.walk(andersen_ncbi_virus):
    for file in files:
        file_name = os.path.join(dirpath, file)
        if ".fasta" in file_name:
        # print(file_name)
            segment_genotype = "_".join(file_name.split("/")[-1].split("_")[0:2])
            fasta_file = fasta_df_complete(file_name, states) # Convert fasta file to dataframe
            andersen_ncbi[segment_genotype] = fasta_file

________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly__

### Collect partial isolates from all parties

In [43]:
# Do all segments, not just HA 

# Check isolate IDs to see if they already exist in Andersen/NCBI
# Match based on year AND partial isolate ID, as some partials may be identical between years

# GISAID partial isolates
gisaid_list = [] 
for gisaid_fasta in segment_fastas:
    # for genotype_gisaid_fasta in gisaid_fasta:
    # print(genotype_gisaid_fasta)
    gisaid_fasta["Partials"] = gisaid_fasta["Isolate_Id"].apply(partial_isolate)
    gisaid_fasta["Year"] = gisaid_fasta["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x).year))
    genotype_gisaid_fasta = gisaid_fasta.drop_duplicates(subset=["Partials", "Year"], keep="first")
    gisaid_list.append(genotype_gisaid_fasta)
    # print(genotype_gisaid_fasta)
        
# NCBI_Virus/Andersen partial isolates    
andersen_ncbi_genotypes = {}
for key in andersen_ncbi:
    andersen_ncbi_fasta = andersen_ncbi[key]

    # Find partial Isolate IDs -- humans and non-humans have different locations for isolates
    andersen_ncbi_fasta_nonhuman = andersen_ncbi_fasta[andersen_ncbi_fasta["Host_Type"] != "human"]
    andersen_ncbi_fasta_nonhuman["Partials"] = andersen_ncbi_fasta_nonhuman["Isolate_Id"].apply(partial_isolate) # For non-humans, apply partial function to isolate ids

    andersen_ncbi_fasta_human = andersen_ncbi_fasta[andersen_ncbi_fasta["Host_Type"] == "human"]
    andersen_ncbi_fasta_human["Partials_Prep"] = andersen_ncbi_fasta_human["Isolate_Name"].apply(lambda x: x.split("/")[-2]) # For humans, apply partial function to location, because isolate comes earlier
    andersen_ncbi_fasta_human["Partials"] = andersen_ncbi_fasta_human["Partials_Prep"].apply(partial_isolate)
    # andersen_ncbi_fasta["Partials"] = andersen_ncbi_fasta["Isolate_Id"].apply(partial_isolate)
    # print("Andersen:", andersen_ncbi_fasta["Partials"])

    # Concatenate humans and non-humans
    andersen_ncbi_fasta = pd.concat([andersen_ncbi_fasta_human, andersen_ncbi_fasta_nonhuman])
    print(andersen_ncbi_fasta)

    # Get year and segment
    andersen_ncbi_fasta["Year"] = andersen_ncbi_fasta["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year) if len(x) > 0 else x)
    andersen_ncbi_fasta["Segment"] = key.split("_")[-1]
    # andersen_ncbi_fasta = andersen_ncbi_fasta.drop_duplicates(subset=["Partials", "Year"], keep="first")
    # print(andersen_ncbi_fasta)

    if andersen_ncbi_fasta["Genotype"].values[0] not in andersen_ncbi_genotypes.keys(): # If we haven't already seen this genotype
        andersen_ncbi_genotypes[andersen_ncbi_fasta["Genotype"].values[0]] = andersen_ncbi_fasta # Add fasta to genotype dictionary
        # print(andersen_ncbi_fasta)
    # break 


                                                 Header  \
3983  SRR31840198|A/Washington/OR/24-037325-011/2024...   
0     SRR28752446|A/blackbird/Texas/24-008354-001/20...   
1     SRR28752447|A/cattle/Texas/24-009108-005/2024|...   
2     SRR28752448|A/cattle/Texas/24-009108-004/2024|...   
3     SRR28752449|A/cattle/Texas/24-009108-003/2024|...   
...                                                 ...   
5166  SRR29281472|A/Turkey/Minnesota/24-014759-001-o...   
5167  SRR29281472|A/Turkey/Minnesota/24-014759-002-o...   
5168  SRR29281472|A/Turkey/Minnesota/24-014759-003-o...   
5169  SRR29281472|A/Turkey/Minnesota/24-014760-001-o...   
5170  SRR29281472|A/Turkey/Minnesota/24-014760-002-o...   

                  Isolate_Id                                    Isolate_Name  \
3983           24-037325-011              A/Washington/OR/24-037325-011/2024   
0              24-008354-001            A/blackbird/Texas/24-008354-001/2024   
1              24-009108-005               A/cattle

### Throw away duplicate isolates from GISAID

In [44]:

throw_away = {} # Dictionary of isolates to throw

counter = 0
for gisaid_genotype in gisaid_list: # Each dataframe is unique in genotype
    print(gisaid_genotype)
    if len(gisaid_genotype["Genotype"]) > 0:
        genotype = gisaid_genotype["Genotype"].values[0]
        andersen_ncbi_fasta = pd.DataFrame()
        if genotype in andersen_ncbi_genotypes.keys(): # If it's in Andersen/NCBI_Virus and we haven't seen it before here
            andersen_ncbi_fasta = andersen_ncbi_genotypes[genotype] # Get the dataframe with the same genotype
            gisaid_duplicates = gisaid_genotype[gisaid_genotype.duplicated(["Partials", "Year"], keep=False)]
            # print(gisaid_duplicates)
            andersen_ncbi_fasta_duplicates = andersen_ncbi_fasta[andersen_ncbi_fasta.duplicated(["Partials", "Year"], keep=False)]
            to_throw = pd.concat([gisaid_genotype, andersen_ncbi_fasta])[pd.concat([gisaid_genotype, andersen_ncbi_fasta]).duplicated(["Partials", "Year"], keep=False)]
            
            between_duplicates = []
            for t in to_throw["Identifier"].values:
                if t not in gisaid_duplicates and t not in andersen_ncbi_fasta_duplicates:
                    # print(t)
                    between_duplicates.append(t)
            if genotype in throw_away.keys(): # If there's already a genotype
                throw_away[genotype] += between_duplicates
            else:
                throw_away[genotype] = between_duplicates
        else: # If it's a genotype not seen in Andersen/NCBI_Virus, don't throw it
            print("GISAID genotype:", gisaid_genotype)
            throw_away[genotype] = []
    counter += 1

for key in throw_away:
    print(key)
    print(len(throw_away[key]))

kept_seqs = []
print(len(segment_fastas))
for gisaid_df in segment_fastas:
    # print(len(genotype_group))
    # for gisaid_df in genotype_group:
        # print(gisaid_df)
    if len(gisaid_df["Genotype"].dropna()) > 0: # If there are sequences
        genotype = gisaid_df["Genotype"].values[0]
        # print(len(genotype_seq_keep[genotype]))
        # gisaid_df_new = gisaid_df[gisaid_df['Identifier'].isin(genotype_seq_keep[genotype])]

        # Check how much was thrown away
        if genotype in throw_away:
            to_throw = throw_away[genotype]
            print("Original length:", genotype, len(gisaid_df))
            gisaid_df_new = gisaid_df[~gisaid_df['Identifier'].isin(to_throw)]
            print("New length:", len(gisaid_df_new))
            kept_seqs.append(gisaid_df_new)
        # else:


# print(len(kept_seqs))

                                                  Header  Isolate_Id  \
2913   EPI_ISL_20307361|A/dairy_cow/California/02121-...   02121-017   
2921   EPI_ISL_20307360|A/dairy_cow/California/02121-...   02121-014   
3073   EPI_ISL_20307357|A/dairy_cow/California/026426...  026426-006   
3081   EPI_ISL_20307356|A/dairy_cow/California/026426...  026426-005   
3089   EPI_ISL_20307359|A/dairy_cow/California/02121-...   02121-010   
3097   EPI_ISL_20307358|A/dairy_cow/California/02121-...   02121-001   
3121   EPI_ISL_20307355|A/dairy_cow/California/026426...  026426-003   
3129   EPI_ISL_20307354|A/dairy_cow/California/026029...  026029-005   
10305  EPI_ISL_20307633|A/dairy_cow/Nebraska/025803-0...  025803-001   
10313  EPI_ISL_20307632|A/dairy_cow/Nebraska/025430-0...  025430-001   

                                 Isolate_Name Subtype Segment    Location  \
2913    A/dairy_cow/California/02121-017/2025    H5N1     PB2  California   
2921    A/dairy_cow/California/02121-014/2025    H5N1

### Create animal reference if needed 

In [45]:
# Find animals to sort, if needed

os.chdir(references)

# Flatten unique_animals
every_unique_animal = []
for l in unique_animals_all:
    for animal in l:
        every_unique_animal.append(animal)

# Rename host type

unique_animals_set = list(set(every_unique_animal))
animals_df = pd.DataFrame(columns=["wild_avian", "domestic_avian", "cattle", "feline", "other_mammal", "human", "pet_food", "other"])
animals_df["other"] = unique_animals_set # to sort

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

print(common_animals)
print(len(common_animals))

different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

print(animals_ref)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer
else:
    number_of_times_to_add_nan = len(different_animals) - len(animals_df)
    nan_row = pd.DataFrame([[np.nan] * len(animals_df.columns)], columns=animals_df.columns)
    for i in range(number_of_times_to_add_nan):
        animals_df = pd.concat([animals_df, nan_row], ignore_index=True)

animals_df["new"] = (different_animals)

print(animals_df)

os.chdir(references)
animals_df.to_csv("animals_ref_to_sort.csv")

['snowy_owl', 'northern_shoveler', 'nevada', 'merganser', 'dunlin', 'parrot', 'american_blue_winged_teal', 'american_goshawk', 'fish_crow', 'skunk', 'cascade_duck', 'rock_goose', 'black_brant', 'mexico', 'procellaria_aequinoctialis', 'great_horned_owl', 'turkey', 'raccoon', 'mottled_duck', 'antarctic_petrel', 'blue_winged_teal', 'house_mouse', 'backyard_turkey', 'elegant_tern', 'swainsons_hawk', 'green_winged_teal', 'chinese_ringneck_pheasant', 'american_black_duck', 'bufflehead', 'chukar', "franklin's_gull", 'magpie', 'northern_pintail', 'black_turnstone', 'house_fly', 'burmeisters_porpoise', 'south_american_sea_lion', 'mixed', 'black_crowned_night-heron', 'northern_fulmar', "cooper'ss_hawk", 'pinniped', 'american_raven', 'humboldt_penguin', 'trumpeter_swan', 'canvasback', 'long_eared_owl', 'harriss_hawk', 'washington', 'antarctic_shag', 'mergus', 'pelecanus_thagus', 'canada_goose', 'dovekie', 'sandwich_tern', 'nene_goose', 'red_fox', 'savannah_cat', 'atlantic_white-sided_dolphin', 'm

In [46]:
# Ensure that user checks if there are any new animals

input("Check animals output. Afterwards, press ESCAPE to continue.")

''

## Merge all fastas into different files -- 8 segments * X genotypes ##

In [ ]:
# 16 files needed
huge_fasta = pd.DataFrame()

for fastas in kept_seqs: # 7 batches
    # print(fastas.columns)
    # print(len(fastas))
    # break
    # for f in fastas: # 16 files per batch 
        # print(f)
        # break 

    # Find Clade
    metadata_fasta = pd.DataFrame()
    for metadata in all_metadata_files:
        metadata_fasta = pd.concat([metadata_fasta, metadata])
    
    metadata_fasta["Identifier"] = metadata_fasta["Isolate_Id"]
    metadata_fasta_concat = fastas.merge(metadata_fasta, on="Identifier", how="left")

    huge_fasta = pd.concat([huge_fasta, metadata_fasta_concat])

print(huge_fasta.columns)



# Now separate huge_fasta into 16 fastas
big_fastas = []

# print(huge_fasta)

# genotypes.append("Unassigned")
for gen in genotypes:
    # print(gen)
    big_fasta = huge_fasta[huge_fasta["Genotype_x"] == gen]
    # print(gen)
    for seg in unique_segments:
        seg_specific_fasta = big_fasta[big_fasta["Segment"] == seg]
        big_fastas.append(seg_specific_fasta)

print(big_fasta)



Index(['Header', 'Isolate_Id_x', 'Isolate_Name_x', 'Subtype_x', 'Segment',
       'Location_x', 'Geo_Location', 'Date Collected', 'Species', 'Sequence',
       'Identifier', 'Host_Type', 'Genotype_x', 'New_Name', 'Partials', 'Year',
       'Isolate_Id_y', 'PB2 Segment_Id', 'PB1 Segment_Id', 'PA Segment_Id',
       'HA Segment_Id', 'NP Segment_Id', 'NA Segment_Id', 'MP Segment_Id',
       'NS Segment_Id', 'HE Segment_Id', 'P3 Segment_Id', 'Isolate_Name_y',
       'Subtype_y', 'Genotype_y', 'Lineage', 'Clade', 'Pathogenicity',
       'Passage_History', 'Location_y', 'Host', 'Isolate_Submitter',
       'Submitting_Sample_Id', 'Publication', 'Originating_Sample_Id',
       'Collection_Date', 'Note', 'Update_Date', 'Submission_Date',
       'Antigen_Character', 'Animal_Vaccin_Product',
       'Adamantanes_Resistance_geno', 'Oseltamivir_Resistance_geno',
       'Zanamivir_Resistance_geno', 'Peramivir_Resistance_geno',
       'Other_Resistance_geno', 'Adamantanes_Resistance_pheno',
       'Os

In [50]:
print(huge_fasta[huge_fasta["Genotype_x"] == "Unassigned"][["Identifier", "Genotype_x", "Genotype_y", "Clade", "Subtype_x", "Pathogenicity"]])

# Now that we have 16 fastas, write the files
for fasta in big_fastas:

    # Create a dictionary to create a file
    # fasta_df = fasta[["New_Name", "Sequence", "Clade"]]
    # print(fasta_df[fasta_df["Clade"].isna() == False])
    # break
    fasta["New_Name_Clade_Path"] = fasta["New_Name"].apply(lambda x: x.split("\n")[0]) # + fasta["Clade"].apply(lambda x: "/" + x if x == x else "") + fasta["Pathogenicity"].apply(lambda x: "/" + x if x == x else "")
    # print(fasta_df["New_Name_Clade"])
    print("Original length:", len(fasta))
    fasta = fasta.drop_duplicates(subset="Identifier", keep="last")
    print("New length:", len(fasta))
    # break 
    fasta_df = fasta[["New_Name_Clade_Path", "Sequence"]]
    fasta_dict = pd.Series(fasta_df.Sequence.values,index=fasta_df.New_Name_Clade_Path).to_dict()
    # print(fasta["Genotype"])
    # Create fasta file 
    try: 
        output_path = gisaid_files + fasta["Genotype_x"].values[0] + "_" + fasta["Segment"].values[0] + "_" + start_date + "--" + end_date + ".fasta" # Genotype and Segment should all be the same
        output_file = open(output_path, "w")
        for item in fasta_dict.keys():
            # print(item)
            value = fasta_dict[item] + "\n"
            # print(value)
            item = item.replace(" ", "_")
            
            # elif "EPI_ISL_20151596" in item:
            #     item = "EPI_ISL_20151596|A/Brown_Skua/Gough_Island/047354/2024|H5N1|Antarctica|2024-09-20|wild_avian|B3.2"
            output_file.write(item + "\n")
            output_file.write(value)
        print("Succeeded in finding results for genotype: ", fasta["Genotype_x"].values[0])
        output_file.close()
    except:
        # print(fasta["Genotype"])
        # print(fasta)
        print("Could not find any results for genotype.")
        # continue

print(len(huge_fasta))
print(len(big_fastas[0]))

Empty DataFrame
Columns: [Identifier, Genotype_x, Genotype_y, Clade, Subtype_x, Pathogenicity]
Index: []
Original length: 435
New length: 435
Succeeded in finding results for genotype:  B3.13
Original length: 435
New length: 435
Succeeded in finding results for genotype:  B3.13
Original length: 435
New length: 435
Succeeded in finding results for genotype:  B3.13
Original length: 435
New length: 435
Succeeded in finding results for genotype:  B3.13
Original length: 435
New length: 435
Succeeded in finding results for genotype:  B3.13
Original length: 435
New length: 435
Succeeded in finding results for genotype:  B3.13
Original length: 435
New length: 435
Succeeded in finding results for genotype:  B3.13
Original length: 435
New length: 435
Succeeded in finding results for genotype:  B3.13
Original length: 1265
New length: 1265
Succeeded in finding results for genotype:  D1.1
Original length: 1265
New length: 1265
Succeeded in finding results for genotype:  D1.1
Original length: 1265
N

## Concatenate to Andersen_NCBI files and save

In [74]:
# Save metadata
os.chdir(andersen_ncbi_virus)
andersen_ncbi_virus_metadata = pd.read_csv("NCBI_Virus_Andersen_" + date_range + "_metadata.csv")
andersen_ncbi_virus_metadata = andersen_ncbi_virus_metadata[["Accession", "Host", "Collection_Date", "Isolate", "Serotype", "Segment", "Genotype", "Host_Type", "Years", "Geo_Location_Abrv", "Name"]]
andersen_ncbi_virus_metadata = andersen_ncbi_virus_metadata.reset_index(drop=True)



# Rename columns
metadata_fasta_csv = metadata_fasta_concat.rename(columns={"Identifier":"Accession", "Isolate_Name_x":"Isolate", "Subtype_x":"Serotype", "Genotype_x":"Genotype", "Year":"Years", "Geo_Location":"Geo_Location_Abrv", "New_Name":"Name"})
metadata_fasta_csv = metadata_fasta_csv[["Accession", "Host", "Collection_Date", "Isolate", "Serotype", "Segment", "Genotype", "Host_Type", "Years", "Geo_Location_Abrv", "Name"]]
metadata_fasta_csv["Isolate"] = metadata_fasta_csv["Isolate"].apply(lambda x: x.split("/")[-2])
segments_map = {"PB2":1, "PB1":2, "PA":3, "HA":4, "NP":5, "NA":6, "MP":7, "NS":8} # Name segments 
metadata_fasta_csv["Segment"] = metadata_fasta_csv["Segment"].map(segments_map)
metadata_fasta_csv = metadata_fasta_csv.reset_index(drop=True)

print(andersen_ncbi_virus_metadata.columns)
print(metadata_fasta_csv.columns)

os.chdir(complete_files)
metadata_all = pd.concat([andersen_ncbi_virus_metadata, metadata_fasta_csv], ignore_index=True)
metadata_all.to_csv("combined_metadata.csv")

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\5\ipykernel_35540\1810407587.py:3: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  andersen_ncbi_virus_metadata = pd.read_csv("NCBI_Virus_Andersen_" + date_range + "_metadata.csv")


Index(['Accession', 'Host', 'Collection_Date', 'Isolate', 'Serotype',
       'Segment', 'Genotype', 'Host_Type', 'Years', 'Geo_Location_Abrv',
       'Name'],
      dtype='object')
Index(['Accession', 'Host', 'Collection_Date', 'Isolate', 'Serotype',
       'Segment', 'Genotype', 'Host_Type', 'Years', 'Geo_Location_Abrv',
       'Name'],
      dtype='object')


In [ ]:
# Concat
os.chdir(complete_files)
segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]

common_genotypes = set()

filenames_ncbi_andersen = []
# File names NCBI_Virus_Andersen
for dirpath, dirs, files in os.walk(andersen_ncbi_virus): # Find the fasta file
    for file in files:
        file_name = os.path.join(dirpath, file) # Get file name
        filenames_ncbi_andersen.append(file_name)
        if ".csv" in file_name: # If metadata
            metadata_ncbi_virus_andersen = pd.read_csv(file_name)
            shutil.copy(file_name, complete_files)
            # metadata_fasta_concat.to_csv("GISAID_metadata.csv")
    break 

filenames_gisaid = []
# File names NCBI_Virus_Andersen
for dirpath, dirs, files in os.walk(gisaid_files): # Find the fasta file
    for file in files:
        file_name = os.path.join(dirpath, file) # Get file name
        filenames_gisaid.append(file_name)
    break 

for a_file in filenames_gisaid:
    a_file_name = "_".join(a_file.split("/")[-1].split("_")[:2])
    print(a_file_name)
    for nv_file in filenames_ncbi_andersen:
        nv_file_name = "_".join(nv_file.split("/")[-1].split("_")[:2])
        print(nv_file_name)
        if a_file_name == nv_file_name and ".fasta" in a_file_name:
            common_genotypes.add(a_file_name)
            filenames = [a_file, nv_file]
            with open(complete_files + a_file_name + "_" + date_range + ".fasta", 'w') as outfile:
                for fname in filenames:
                    with open(fname) as infile:
                        for line in infile:
                            outfile.write(line)
            infile.close()
    outfile.close()

for a_file in filenames_ncbi_andersen:
    partial_filename_a = "_".join(a_file.split("/")[-1].split("_")[:2])
    print(partial_filename_a)
    # If genotype not found in one of the datasets, include it as well
    if partial_filename_a not in common_genotypes and ".fasta" in a_file:
        for segment in segments:
            with open(complete_files + partial_filename_a + "_" + date_range + ".fasta", 'w') as outfile2:
                with open(a_file) as infile2:
                    for line in infile2:
                        outfile2.write(line)
                    infile2.close()
                outfile2.close()

for a_file in filenames_gisaid:
    partial_filename_a = "_".join(a_file.split("/")[-1].split("_")[:2])
    print(partial_filename_a)
    # If genotype not found in one of the datasets, include it as well
    if partial_filename_a not in common_genotypes and ".fasta" in a_file:
        for segment in segments:
            with open(complete_files + partial_filename_a + "_" + date_range + ".fasta", 'w') as outfile3:
                with open(a_file) as infile3:
                    for line in infile3:
                        outfile3.write(line)
                    infile3.close()
                outfile3.close()

B3.13_HA
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2
D1.3_HA
D1.3_MP
D1.3_NA
D1.3_NP
D1.3_NS
D1.3_PA
D1.3_PB1
D1.3_PB2
NCBI_Virus
B3.13_MP
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2
D1.3_HA
D1.3_MP
D1.3_NA
D1.3_NP
D1.3_NS
D1.3_PA
D1.3_PB1
D1.3_PB2
NCBI_Virus
B3.13_NA
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2
D1.3_HA
D1.3_MP
D1.3_NA
D1.3_NP
D1.3_NS
D1.3_PA
D1.3_PB1
D1.3_PB2
NCBI_Virus
B3.13_NP
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2
D1.3_HA
D1.3_MP
D1.3_NA
D1.3_NP
D1.3_NS
D1.3_PA
D1.3_PB1
D1.3_PB2
NCBI_Virus
B3.13_NS
B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
D1.1_HA
D1.1_

In [54]:
# Create new labels FOR D1.3 ONLY

# Create new labels
os.chdir(home)
new_county_info = pd.read_excel("Positive Case Info  Summary (version 1).xlsx")
og_county_info = pd.read_csv("Positive Case Info = Summary Counties.csv")
sra_info = pd.read_excel("OH-IN_poultry_clade_matched_seq_metadata (version 1).xlsx")

sra_info["Event ID"] = sra_info["SRA Event ID"]

# print(sra_info)
# print(og_county_info["County"])
# print(county_info["HPAI Case Description"])
# print(county_info["Event ID"])

med_county_info = new_county_info.merge(sra_info, on=["Event ID"])

county_info = med_county_info.merge(og_county_info, on=["Event ID", "Site Owner", "Collection Date"], how="left")
# print(county_info)
# print(county_info["County"])
# new_county_info["County"] = np.where(new_county_info["County"] == "",  new_county_info["County"])
county_info["County"] = county_info["HPAI Case Description"].apply(lambda x: # og_county_info.loc[county_info["HPAI Case Description"].str.contains('_'.join(x.split(' ')[:-1])), 'County'] if len(og_county_info.loc[county_info["HPAI Case Description"].str.contains('_'.join(x.split(' ')[:-1])), 'County']) > 0 else 
                                                                   '_'.join(x.split(' ')[:-1]) + "_County_" + og_county_info[og_county_info["County"].str.contains("_".join(x.split(' ')[:-1]))]["County"].values[0].split("_")[-1] if x == x else x)
# print(county_info["Collection Date"])

# print(county_info[county_info["County"].notna()]["County"])

label_mapping = {}
for id in county_info["SRA Accession"].values:
    print(id)
    if id == id and county_info[county_info["SRA Accession"] == id]["County"].values[0] == county_info[county_info["SRA Accession"] == id]["County"].values[0]: # If not nan
        label_mapping[id] = county_info[county_info["SRA Accession"] == id]["County"].iloc[0] + "|" + county_info[county_info["SRA Accession"] == id]["Site Owner"].iloc[0].replace(" ", "_")

print(len(label_mapping))
print(label_mapping)

for dirpath, dirs, files in os.walk(complete_files):
    for file in files:
        file_name = os.path.join(dirpath, file)
        if "D1.3" in file_name:
            output_file = open(file_name, "r+")
            with open(file_name) as f:
                lines = f.readlines()
                for line in lines:
                    for key in label_mapping:
                        # break 
                        if key in line and len(line.split("|")) < 8: 
                            print(line.split("|")[-3])
                            if dateutil.parser.parse(line.split("|")[-3], default=dateutil.parser.parse("2000-01-01")).month == dateutil.parser.parse("2000-01-01").month and dateutil.parser.parse(line.split("|")[-3], default=dateutil.parser.parse("2000-01-01")).day == dateutil.parser.parse("2000-01-01").day: # If no collection date in line
                                date = county_info[county_info["SRA Accession"] == key]["Collection Date"].apply(lambda x: dateutil.parser.parse(str(x))).values[0]
                                # print(date)
                                split_line = line.split("|")
                                split_line[-3] = dateutil.parser.parse(str(date)).strftime("%Y-%m-%d")
                                line = ("|").join(split_line)
                                print(line)
                                print("date changed")
                            if line.split("|")[3] == "USA":
                                split_line = line.split("|")
                                split_line[3] = "USA-" + county_info[county_info["SRA Accession"] == key]["County"].apply(lambda x: x.split("_")[-1]).values[0]
                                line = ("|").join(split_line)
                            line = line.replace("\n", "") + "|" + label_mapping[key] + "\n"
                            print(line)
                            print("changed")
                    output_file.write(line)
                f.close()
            output_file.close()
    break 

SRR32654191
SRR32654192
SRR32654193
SRR32654195
SRR32654191
SRR32654192
SRR32654193
SRR32654195
SRR32654191
SRR32654192
SRR32654193
SRR32654195
SRR32654191
SRR32654192
SRR32654193
SRR32654195
SRR32415174
SRR32415176
SRR32415177
SRR32415178
SRR32415174
SRR32415176
SRR32415177
SRR32415178
SRR32415174
SRR32415176
SRR32415177
SRR32415178
SRR32415174
SRR32415176
SRR32415177
SRR32415178
SRR31959950
SRR31959951
SRR31959950
SRR31959951
SRR32125543
SRR32125544
SRR32125543
SRR32125544
SRR32226901
SRR32226902
SRR32226904
SRR32226915
SRR32226901
SRR32226902
SRR32226904
SRR32226915
SRR32226901
SRR32226902
SRR32226904
SRR32226915
SRR32226901
SRR32226902
SRR32226904
SRR32226915
SRR32226934
SRR32226935
SRR32226936
SRR32226938
SRR32226934
SRR32226935
SRR32226936
SRR32226938
SRR32226934
SRR32226935
SRR32226936
SRR32226938
SRR32226934
SRR32226935
SRR32226936
SRR32226938
SRR32226929
SRR32226930
SRR32226929
SRR32226930
SRR32226927
SRR32226928
SRR32226927
SRR32226928
SRR32254627
SRR32254628
SRR32254627
SRR3